In [1]:
import asyncio
import json
import websockets

In [2]:
import requests
def create_room():
    responce = requests.post("http://127.0.0.1:8000/game/rooms")
    if responce.status_code == 200:
        return responce.json()["room_id"]
    return None

ALL_CREATED_CLIENTS = []
class TestClient:
    def __init__(self, room_id):
        self.room_id = room_id
        self.ws = None
        self.receiver_task = None
        self.connector = asyncio.Queue(maxsize=1)

    def get_connector(self):
        return self.connector

    async def connect(self):
        self.ws = await websockets.connect(
            f"ws://127.0.0.1:8000/game/{self.room_id}"
        )

        self.receiver_task = asyncio.create_task(
            self.receiver()
        )

        ALL_CREATED_CLIENTS.append(self)

    async def receiver(self):
        async for message in self.ws:
            if self.connector.full():
                self.connector.get_nowait()
            self.connector.put_nowait(json.loads(message))

    async def send(self, data):
        await self.ws.send(json.dumps(data))

    async def close(self):
        self.receiver_task.cancel()
        await self.ws.close()



In [3]:
from IPython.display import display, HTML


class GameMapViewer:
    def __init__(self):
        self.symbols = {
            0: ".",
            2: "F",
            3: "#",
            4: "O",
        }

        self.output = display(
            HTML("<pre>Starting...</pre>"),
            display_id=True
        )

    def update(self, snapshot):
        current_map = [
            row.copy()
            for row in snapshot["map"]
        ]

        for snake in snapshot["snakes"].values():
            for x, y in snake["body"]:
                current_map[y][x] = "S"

            if snake["alive"]:
                x, y = snake["body"][-1]
                current_map[y][x] = "H"

        text = "\n".join(
            "".join(
                self.symbols.get(tile, "?")
                for tile in row
            )
            for row in reversed(current_map)
        )

        self.output.update(
            HTML(f"<pre>{text}</pre>")
        )

In [7]:
room_id = create_room()
p1 = TestClient(room_id)
await p1.connect()

In [5]:
viewer = GameMapViewer()

connector = p1.get_connector()

async def watch(connector):
    while True:
        snapshot = await connector.get()
        viewer.update(snapshot)

task = asyncio.create_task(watch(connector))

In [6]:
await p1.send({"command":"up"})

In [8]:
connector = p1.get_connector()
await connector.get()

{'snakes': {'53b1c743af644871a13efc52cb9c3eb6': {'hp': 5,
   'alive': True,
   'direction': 0,
   'body': [[5, 7]]}},
 'world': [[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3],
  [3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3],
  [3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3],
  [3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3],
  [3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3],
  [3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3],
  [3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3],
  [3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3],
  [3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3],
  [3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3],
  [3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3],
  [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]]}